# Generate daily SEACOFS eddy-overlay frames

Generate one PNG per available model day for later assembly into an animation. Each frame follows `eddy_velocity_overlay.ipynb`: rotated surface speed and arrows, processed eddy centres and IDs, and the fitted $R_c^2/2$ ESP contour.

The NetCDF files are processed one file at a time, figures are closed immediately, existing frames can be skipped, and a CSV manifest records the animation order.

In [ ]:
from pathlib import Path
import sys

import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import netCDF4 as nc
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from seacofs_eddy_dataset.config import load_config
from seacofs_eddy_dataset.core.grid import read_reference_grid
from seacofs_eddy_dataset.core.velocity import rotate_uv
from seacofs_eddy_dataset.stages.detection import find_model_files, reference_grid_path
plt.ioff()


## Controls

In [ ]:
CONFIG_PATH = PROJECT_ROOT / "config" / "local.yaml"
config = load_config(CONFIG_PATH)

grid = read_reference_grid(reference_grid_path(config))
PROCESSED_PATH = config.output_root / "processed" / "eddy_dataset_processed.parquet"
FRAME_DIR = config.output_root / "daily_overlay_frames"
MANIFEST_PATH = FRAME_DIR / "frame_manifest.csv"

# Set either limit to an integer Day or leave as None to render the full record.
START_DAY = None
END_DAY = None

OVERWRITE = False
NUM_LABEL = True
SHOW_QUIVER = True
QUIVER_STEP = 12
QUIVER_WIDTH = 0.0015
FIGSIZE = (9, 10)
DPI = 150

# A single limit prevents colour flicker in the animation. Leave as None to
# estimate it from sampled days, or provide a fixed speed in m s^-1.
SPEED_VMAX = None
SPEED_PERCENTILE = 99.0
SAMPLES_PER_FILE = 3

FRAME_DIR.mkdir(parents=True, exist_ok=True)
FRAME_DIR


## Load eddies and source files

In [ ]:
if not PROCESSED_PATH.exists():
    raise FileNotFoundError(f"Processed dataset not found: {PROCESSED_PATH}")
df = pd.read_parquet(PROCESSED_PATH)
df["Date"] = pd.to_datetime(df["Date"]).dt.normalize()
df = df.sort_values(["Day", "Eddy"]).reset_index(drop=True)
eddies_by_day = {int(day): group.copy() for day, group in df.groupby("Day", sort=False)}

source_files = find_model_files(config)
if not source_files:
    raise FileNotFoundError(f"No model files found under {config.model_root}")
print(f"Processed eddies: {len(df):,} rows, {df.Eddy.nunique():,} tracks")
print(f"Source files: {len(source_files)}")
print(f"Frame directory: {FRAME_DIR}")


## Determine one colour scale for all frames

In [ ]:
def estimate_speed_vmax(paths, percentile=SPEED_PERCENTILE, samples_per_file=SAMPLES_PER_FILE):
    sampled_speeds = []
    for path in paths:
        with nc.Dataset(path) as dataset:
            nt = len(dataset.variables["ocean_time"])
            sample_count = min(samples_per_file, nt)
            indices = np.linspace(0, nt - 1, sample_count, dtype=int)
            for index in np.unique(indices):
                u = dataset["u_eastward"][index, -1, :, :].T
                v = dataset["v_northward"][index, -1, :, :].T
                u_rot, v_rot = rotate_uv(u, v, grid.angle)
                speed = np.hypot(u_rot, v_rot)
                sampled_speeds.append(speed[np.isfinite(speed)].ravel())
    if not sampled_speeds:
        raise ValueError("No finite SEACOFS velocities were found while estimating SPEED_VMAX.")
    return float(np.percentile(np.concatenate(sampled_speeds), percentile))

if SPEED_VMAX is None:
    SPEED_VMAX = estimate_speed_vmax(source_files)
print(f"Shared speed colour limit: 0 to {SPEED_VMAX:.3f} m s^-1")


## Frame renderer

In [ ]:
def render_frame(u_rot, v_rot, df_day, day, output_path):
    speed = np.hypot(u_rot, v_rot)
    fig, ax = plt.subplots(figsize=FIGSIZE, constrained_layout=True)
    image = ax.pcolormesh(grid.X_grid, grid.Y_grid, speed, shading="auto", vmin=0, vmax=SPEED_VMAX, cmap="Blues_r")
    fig.colorbar(image, ax=ax, label=r"Rotated surface current speed (m s$^{-1}$)", shrink=0.7)

    if SHOW_QUIVER:
        step = max(1, int(QUIVER_STEP))
        ax.quiver(grid.X_grid[::step, ::step], grid.Y_grid[::step, ::step], u_rot[::step, ::step], v_rot[::step, ::step], color="0.2", alpha=0.55, pivot="mid", width=QUIVER_WIDTH)

    colours = {"AE": "red", "CE": "cyan"}
    for row in df_day.itertuples(index=False):
        colour = colours.get(row.Cyc, "white")
        ax.scatter(row.xc, row.yc, color=colour, edgecolor="black", linewidth=0.8, s=20, zorder=10)
        if np.all(np.isfinite([row.q11, row.q12, row.q22, row.Rc])):
            Q = np.array([[row.q11, row.q12], [row.q12, row.q22]], dtype=float)
            dx, dy = grid.X_grid - row.xc, grid.Y_grid - row.yc
            rho2 = Q[0, 0] * dx**2 + 2 * Q[0, 1] * dx * dy + Q[1, 1] * dy**2
            ax.contour(grid.X_grid, grid.Y_grid, rho2, levels=[row.Rc**2 / 2], colors=[colour], linewidths=1.5, zorder=9)
        if NUM_LABEL:
            ax.annotate(str(int(row.Eddy)), (row.xc, row.yc), textcoords="offset points", xytext=(3, 3), fontsize=8, color="white", weight="bold", path_effects=[pe.withStroke(linewidth=2, foreground="black")], zorder=11)

    lon_levels = np.arange(np.ceil(np.nanmin(grid.lon_rho) / 2) * 2, np.nanmax(grid.lon_rho), 2)
    lat_levels = np.arange(np.ceil(np.nanmin(grid.lat_rho) / 2) * 2, np.nanmax(grid.lat_rho), 2)
    if lat_levels.size:
        contours = ax.contour(grid.X_grid, grid.Y_grid, grid.lat_rho, levels=lat_levels, colors="black", linewidths=0.5, alpha=0.7)
        ax.clabel(contours, fmt=lambda value: f"{abs(value):.0f}°S", fontsize=8)
    if lon_levels.size:
        contours = ax.contour(grid.X_grid, grid.Y_grid, grid.lon_rho, levels=lon_levels, colors="black", linewidths=0.5, alpha=0.7)
        ax.clabel(contours, fmt=lambda value: f"{value:.0f}°E", fontsize=8)

    ae_count = int(df_day.Cyc.eq("AE").sum()) if not df_day.empty else 0
    ce_count = int(df_day.Cyc.eq("CE").sum()) if not df_day.empty else 0
    date_label = f" | {df_day.Date.iloc[0]:%Y-%m-%d}" if not df_day.empty and "Date" in df_day else ""
    ax.set(title=f"SEACOFS eddies | Day {day}{date_label} | AE={ae_count}, CE={ce_count}", xlabel="x (km)", ylabel="y (km)", xlim=(grid.x_grid.min(), grid.x_grid.max()), ylim=(grid.y_grid.min(), grid.y_grid.max()), aspect="equal")
    fig.savefig(output_path, dpi=DPI, facecolor="white")
    plt.close(fig)
    return ae_count, ce_count


## Generate frames

Files are named with a zero-padded global frame number followed by the model day, so lexical order is animation order. Rerunning with `OVERWRITE=False` resumes without redrawing completed frames.

In [ ]:
empty_day = df.iloc[0:0].copy()
records = []
frame_number = 0
seen_days = set()

for source_file in source_files:
    with nc.Dataset(source_file) as dataset:
        days = np.rint(np.asarray(dataset.variables["ocean_time"][:].data, dtype=float) / 86400).astype(int)
        for time_index, day in enumerate(days):
            day = int(day)
            if day in seen_days:
                raise ValueError(f"Duplicate model Day across source files: {day}")
            seen_days.add(day)
            if (START_DAY is not None and day < int(START_DAY)) or (END_DAY is not None and day > int(END_DAY)):
                continue

            df_day = eddies_by_day.get(day, empty_day)
            frame_path = FRAME_DIR / f"frame_{frame_number:06d}_day_{day:05d}.png"
            status = "skipped"
            if OVERWRITE or not frame_path.exists():
                u = dataset["u_eastward"][time_index, -1, :, :].T
                v = dataset["v_northward"][time_index, -1, :, :].T
                u_rot, v_rot = rotate_uv(u, v, grid.angle)
                ae_count, ce_count = render_frame(u_rot, v_rot, df_day, day, frame_path)
                status = "written"
            else:
                ae_count = int(df_day.Cyc.eq("AE").sum()) if not df_day.empty else 0
                ce_count = int(df_day.Cyc.eq("CE").sum()) if not df_day.empty else 0

            records.append({"frame": frame_number, "day": day, "source_file": str(source_file), "time_index": time_index, "frame_path": str(frame_path), "AE": ae_count, "CE": ce_count, "status": status})
            frame_number += 1
            if frame_number % 50 == 0:
                print(f"Completed {frame_number:,} frames through Day {day}")

manifest = pd.DataFrame(records).sort_values("frame").reset_index(drop=True)
manifest.to_csv(MANIFEST_PATH, index=False)
print(f"Finished {len(manifest):,} frames. Manifest: {MANIFEST_PATH}")
manifest.tail()


## Generation summary

In [ ]:
pd.Series({"frames": len(manifest), "first_day": manifest.day.min(), "last_day": manifest.day.max(), "written_this_run": manifest.status.eq("written").sum(), "skipped_existing": manifest.status.eq("skipped").sum(), "frame_directory": str(FRAME_DIR), "speed_vmax_m_s": SPEED_VMAX}, name="value").to_frame()
